## Part A — Databricks setup and Bronze layer
1. Create a schema named `retail_fresher`.
2. Create a managed volume named `retail_raw`.
3. Upload the three CSV files into the volume.

In [0]:
import shutil
import os
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

In [0]:
%sql
USE CATALOG retail_pulse;

CREATE SCHEMA IF NOT EXISTS retail_fresher
COMMENT 'This schema holds raw files';

CREATE SCHEMA IF NOT EXISTS retail_bronze
COMMENT 'This schema holds bronze tables';

CREATE SCHEMA IF NOT EXISTS retail_silver
COMMENT 'This schema holds silver tables';

CREATE SCHEMA IF NOT EXISTS retail_gold
COMMENT 'This schema holds gold tables';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS retail_fresher.retail_raw
COMMENT 'This volume holds raw files';

In [0]:
VOLUME_PATH = "/Volumes/retail_pulse/retail_fresher/retail_raw"
BRONZE_SCHEMA = "retail_pulse.retail_bronze"
SILVER_SCHEMA = "retail_pulse.retail_silver"
GOLD_SCHEMA = "retail_pulse.retail_gold"

In [0]:
source_path = "/Workspace/Users/madhulikasawant@gmail.com/retail-pulse-weekly-sales-intelligence/datasets"
file_names = ["customers_500.csv", "products_500.csv", "sales_orders_500.csv"]

for file_name in file_names:
    source_file = os.path.join(source_path, file_name)
    destination_file = os.path.join(VOLUME_PATH, file_name)
    
    print(f"Copying {source_file} to {destination_file}")
    shutil.copy(source_file, destination_file)

display(dbutils.fs.ls(VOLUME_PATH))

4. Read every CSV with PySpark.
5. Keep the initial CSV columns as strings in the Bronze layer.
6. Add `source_file` and `ingestion_timestamp`.
7. Save managed Delta tables:
   - `bronze_customers`
   - `bronze_products`
   - `bronze_sales_orders`


In [0]:
def read_raw_file(file_name):
    file_path = os.path.join(VOLUME_PATH, file_name)
    df = spark.read.option("header", "True").option("inferSchema", False).csv(file_path)
    df_with_metadata = df.withColumns(
        {"source_file": F.lit(file_path), "ingestion_timestamp": F.current_timestamp()}
    )
    return df_with_metadata

def ingest_bronze_data(file_name, table_name):
    raw_df = read_raw_file(file_name)
    raw_df_row_count = raw_df.count()

    raw_df.write.mode("overwrite").saveAsTable(f"{BRONZE_SCHEMA}.{table_name}")
    print(f"Loaded {raw_df_row_count} records into {table_name}")

    bronze_df = spark.table(f"{BRONZE_SCHEMA}.{table_name}")
    bronze_df_row_count = bronze_df.count()
    
    print(f"Validating {bronze_df_row_count} records in {table_name}")
    assert (
        raw_df_row_count == bronze_df_row_count
    ), f"Expected {raw_df_row_count} records in {table_name}, found {bronze_df_row_count}"
    
    print(f"Validating schema for {table_name}")
    assert [(f.name, f.dataType) for f in raw_df.schema] == [
        (f.name, f.dataType) for f in bronze_df.schema
    ], f"Expected schema {raw_df.schema} in {table_name}, found {bronze_df.schema}"

    return bronze_df

bronze_customers_df = ingest_bronze_data("customers_500.csv", "bronze_customers")
bronze_products_df = ingest_bronze_data("products_500.csv", "bronze_products")
bronze_sales_orders_df = ingest_bronze_data("sales_orders_500.csv", "bronze_sales_orders")


## Part B — PySpark transformations
1. Display schema and sample rows.

In [0]:
bronze_customers_df.printSchema()
bronze_customers_df.show()

bronze_products_df.printSchema()
bronze_products_df.show()

bronze_sales_orders_df.printSchema()
bronze_sales_orders_df.show()

2. Trim string columns and standardize upper/lower case where appropriate.
3. Replace missing city values with `Unknown`.
4. Cast dates, timestamps, integers and decimal columns.

In [0]:
bronze_customers_trimmed_df = bronze_customers_df.withColumns({
    "customer_id": F.regexp_extract(F.upper(F.trim(F.col("customer_id"))), r"^C\d{4}", 0),
    "customer_name": F.initcap(F.trim(F.col("customer_name"))),
    "email": F.regexp_extract(F.lower(F.trim(F.col("email"))), r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$", 0),
    "city": F.initcap(F.trim(F.col("city"))),
    "city": F.ifnull(F.col("city"), F.lit("Unknown")),
    "state": F.initcap(F.trim(F.col("state"))),
    "region": F.initcap(F.trim(F.col("region"))),
    "customer_segment": F.initcap(F.trim(F.col("customer_segment"))),
    "signup_date": F.try_to_date(F.trim(F.col("signup_date")), "yyyy-MM-dd"),
    "date_of_birth": F.try_to_date(F.trim(F.col("date_of_birth")), "yyyy-MM-dd"),
    "is_active": F.when(F.trim(F.col("is_active")) == 'Y', F.lit(True)).otherwise(F.lit(False)),
    "loyalty_points": F.trim(F.col("loyalty_points")).try_cast(IntegerType()),
    "updated_at": F.trim(F.col("updated_at")).try_cast("timestamp")
})

bronze_products_trimmed_df = bronze_products_df.withColumns({
    "product_id": F.regexp_extract(F.upper(F.trim(F.col("product_id"))), r"^P\d{4}", 0),
    "product_name": F.initcap(F.trim(F.col("product_name"))),
    "category": F.initcap(F.trim(F.col("category"))),
    "subcategory": F.initcap(F.trim(F.col("subcategory"))),
    "unit_price": F.trim(F.col("unit_price")).try_cast("decimal(10, 2)"),
    "cost_price": F.trim(F.col("cost_price")).try_cast("decimal(10, 2)"),
    "supplier_name": F.initcap(F.trim(F.col("supplier_name"))),
    "stock_quantity": F.trim(F.col("stock_quantity")).try_cast(IntegerType()),
    "launch_date": F.try_to_date(F.trim(F.col("launch_date")), "yyyy-MM-dd"),
    "product_rating": F.trim(F.col("product_rating")).try_cast("decimal(4, 2)"),
    "active_flag": F.when(F.trim(F.col("active_flag")) == 'Y', F.lit(True)).otherwise(F.lit(False))
})

bronze_sales_orders_trimmed_df = bronze_sales_orders_df.withColumns({
    "order_id": F.upper(F.trim(F.col("order_id"))),
    "order_timestamp":F.trim(F.col("order_timestamp")).try_cast("timestamp"),
    "customer_id": F.regexp_extract(F.upper(F.trim(F.col("customer_id"))), r"^C\d{4}", 0),
    "product_id": F.regexp_extract(F.upper(F.trim(F.col("product_id"))), r"^P\d{4}", 0),
    "quantity": F.trim(F.col("quantity")).try_cast(IntegerType()),
    "discount_pct": F.coalesce(F.trim(F.col("discount_pct")).try_cast("decimal(5, 2)"), F.lit(0.00)),
    "payment_method": F.upper(F.trim(F.col("payment_method"))),
    "order_status": F.upper(F.trim(F.col("order_status"))),
    "promised_delivery_date": F.try_to_date(F.trim(F.col("promised_delivery_date")), "yyyy-MM-dd"),
    "actual_delivery_date": F.try_to_date(F.trim(F.col("actual_delivery_date")), "yyyy-MM-dd"),
    "sales_channel": F.upper(F.trim(F.col("sales_channel"))),
    "warehouse_id": F.regexp_extract(F.upper(F.trim(F.col("warehouse_id"))), r"^WH\d{2}", 0)
})

bronze_customers_trimmed_df.printSchema()
bronze_customers_trimmed_df.show()

bronze_products_trimmed_df.printSchema()
bronze_products_trimmed_df.show()

bronze_sales_orders_trimmed_df.printSchema()
bronze_sales_orders_trimmed_df.show()

5. Deduplicate customer profiles using `row_number()` and the latest
   `updated_at`
6. Filter inactive customers.

In [0]:
rank_window = Window.partitionBy("customer_id").orderBy(F.desc("updated_at"))
bronze_customers_deduplicated_df = (
    bronze_customers_trimmed_df.withColumn("rank", F.rank().over(rank_window))
    .where((F.col("rank") == 1) & (F.col("is_active")))
    .drop("rank")
)

display(bronze_customers_deduplicated_df)

7. Remove products with invalid prices or costs.
8. Filter inactive products.

In [0]:
valid_unit_price = F.col("unit_price") > 0
valid_cost_price = F.col("cost_price") > 0
active_products = F.col("active_flag")

bronze_products_filtered_df = bronze_products_trimmed_df.filter(
    valid_unit_price & valid_cost_price & active_products
)

display(bronze_products_filtered_df)

9. Filter orders with quantity less than or equal to zero.
10. Exclude `PENDING` and `CANCELLED` orders from financial analysis.
11. Add:
    - `gross_amount`
    - `discount_amount`
    - `net_amount`
    - `net_sales`
    - `profit_per_unit`
    - `delivery_days`
    - `late_delivery_flag`
    - `order_month`

In [0]:
valid_quantity = F.col("quantity") > 0
valid_order_status = ~F.col("order_status").isin("PENDING", "CANCELLED")
bronze_sales_orders_filtered_df = bronze_sales_orders_trimmed_df.filter(
    valid_quantity & valid_order_status
)

bronze_sales_orders_enriched_df = (
    bronze_sales_orders_filtered_df.join(
        bronze_products_filtered_df, "product_id", "inner"
    )
    .withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
    .withColumn(
        "discount_amount",
        F.round(F.col("gross_amount") * (F.col("discount_pct") / 100), 2),
    )
    .withColumn("net_amount", F.col("gross_amount") - F.col("discount_amount"))
    .withColumn(
        "net_sales",
        F.when(F.col("order_status") == "RETURNED", F.lit(0)).otherwise(
            F.col("net_amount")
        ),
    )
    .withColumn("profit_per_unit", F.col("unit_price") - F.col("cost_price"))
    .withColumn(
        "delivery_days",
        F.datediff(F.col("actual_delivery_date"), F.col("order_timestamp")),
    )
    .withColumn(
        "late_delivery_flag",
        F.when(
            F.col("actual_delivery_date") > F.col("promised_delivery_date"), F.lit(True)
        ).otherwise(F.lit(False)),
    )
    .withColumn("order_month", F.month(F.col("order_timestamp")))
    .withColumn("order_year", F.year(F.col("order_timestamp")))
)

display(bronze_sales_orders_enriched_df)

12. Join customers, products and orders.

In [0]:
bronze_sales_orders_joined_df = bronze_sales_orders_enriched_df.join(bronze_customers_deduplicated_df, "customer_id", "inner")

display(bronze_sales_orders_joined_df)

13. Store Silver managed Delta tables.

In [0]:
bronze_customers_row_count = bronze_customers_deduplicated_df.count()
print(f"Loading {bronze_customers_row_count} rows into {SILVER_SCHEMA}.customers")
(
    bronze_customers_deduplicated_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.customers")
)

bronze_products_row_count = bronze_products_filtered_df.count()
print(f"Loading {bronze_products_row_count} rows into {SILVER_SCHEMA}.products")
(
    bronze_products_filtered_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.products")
)

bronze_sales_orders_final_df = bronze_sales_orders_joined_df.select(
    F.col("order_id"),
    F.col("order_timestamp"),
    F.col("customer_id"),
    F.col("product_id"),
    F.col("quantity"),
    F.col("discount_pct"),
    F.col("payment_method"),
    F.col("order_status"),
    F.col("promised_delivery_date"),
    F.col("actual_delivery_date"),
    F.col("sales_channel"),
    F.col("warehouse_id"),
    F.col("gross_amount"),
    F.col("discount_amount"),
    F.col("net_amount"),
    F.col("net_sales"),
    F.col("profit_per_unit"),
    F.col("delivery_days"),
    F.col("late_delivery_flag"),
    F.col("order_month"),
    F.col("order_year")
)
bronze_sales_orders_row_count = bronze_sales_orders_final_df.count()
print(f"Loading {bronze_sales_orders_row_count} rows into {SILVER_SCHEMA}.sales_orders")
(
    bronze_sales_orders_final_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_SCHEMA}.sales_orders")
)

silver_customers_df = spark.table(f"{SILVER_SCHEMA}.customers")
silver_products_df = spark.table(f"{SILVER_SCHEMA}.products")
silver_sales_orders_df = spark.table(f"{SILVER_SCHEMA}.sales_orders")

silver_customers_row_count = silver_customers_df.count()
print(f"Validating {silver_customers_row_count} rows in {SILVER_SCHEMA}.customers")
assert silver_customers_row_count == bronze_customers_row_count

silver_products_row_count = silver_products_df.count()
print(f"Validating {silver_products_row_count} rows in {SILVER_SCHEMA}.products")
assert silver_products_row_count == bronze_products_row_count

silver_sales_orders_row_count = silver_sales_orders_df.count()
print(f"Validating {silver_sales_orders_row_count} rows in {SILVER_SCHEMA}.sales_orders")
assert silver_sales_orders_row_count == bronze_sales_orders_row_count

14. Create Gold tables:
    - monthly category sales
    - city sales
    - customer value
    - top products by category

In [0]:
%sql
DROP TABLE IF EXISTS retail_gold.monthly_category_sales;

CREATE TABLE retail_gold.monthly_category_sales
USING DELTA 
AS SELECT 
    category,
    order_month,
    order_year,
    ROUND(SUM(net_sales), 2) AS monthly_net_sales
FROM retail_silver.sales_orders AS so 
JOIN retail_silver.products AS p
ON so.product_id = p.product_id
GROUP BY category, order_year, order_month;

SELECT * FROM retail_gold.monthly_category_sales;

In [0]:
%sql 
SELECT * FROM retail_gold.monthly_category_sales;

In [0]:
%sql
DROP TABLE IF EXISTS retail_gold.city_sales;
     
CREATE TABLE retail_gold.city_sales
USING DELTA
AS SELECT
    city,
    ROUND(SUM(net_sales), 2) AS total_net_sales
FROM retail_silver.sales_orders AS so
JOIN retail_silver.customers AS c
ON so.customer_id = c.customer_id
GROUP BY city;

SELECT * FROM retail_gold.city_sales;

In [0]:
%sql
DROP TABLE IF EXISTS retail_gold.customer_value;

CREATE TABLE retail_gold.customer_value 
USING DELTA
AS SELECT
    customer_id,
    ROUND(SUM(net_sales), 2) AS total_net_sales
FROM retail_silver.sales_orders
GROUP BY customer_id;

SELECT * FROM retail_gold.customer_value;

In [0]:
%sql
DROP TABLE IF EXISTS retail_gold.top_products_by_category;

CREATE TABLE retail_gold.top_products_by_category
USING DELTA
AS WITH ranked_products AS (
    SELECT
        p.category,
        so.product_id,
        ROUND(SUM(so.quantity), 2) AS total_quantity,
        ROUND(SUM(so.net_sales), 2) AS total_net_sales,
        ROW_NUMBER() OVER (
            PARTITION BY p.category 
            ORDER BY SUM(so.net_sales) DESC, SUM(so.quantity) ASC
        ) AS rank
    FROM retail_pulse.retail_silver.sales_orders AS so
    JOIN retail_pulse.retail_silver.products AS p
        ON so.product_id = p.product_id
    GROUP BY p.category, so.product_id
)
SELECT
    category,
    product_id,
    total_quantity,
    total_net_sales
FROM ranked_products
WHERE rank <= 5
ORDER BY category, total_net_sales DESC;

SELECT * FROM retail_gold.top_products_by_category;

## Part C — Aggregation and window functions
1. `GROUP BY` category and month.
2. `SUM`, `COUNT`, `AVG`, `MIN` and `MAX`.

In [0]:
display(
    silver_sales_orders_df.join(silver_products_df, "product_id", "inner")
    .filter(F.col("order_status") == "COMPLETED")
    .groupBy(F.col("category"), F.col("order_year"), F.col("order_month"))
    .agg(
        F.round(F.sum(F.col("net_sales")), 2).alias("total_net_sales"),
        F.count(F.col("order_id").alias("total_orders")),
        F.round(F.avg(F.col("net_sales")), 2).alias("avg_net_sales"),
        F.round(F.min(F.col("net_sales")), 2).alias("min_net_sales"),
        F.round(F.max(F.col("net_sales")), 2).alias("max_net_sales"),
    )
)

3. `row_number()` to keep the latest customer profile.

In [0]:
row_window = Window.partitionBy(F.col("customer_id")).orderBy(F.col("updated_at"))
display(
    silver_customers_df.withColumn("row_no", F.row_number().over(row_window))
    .filter(F.col("row_no") == 1)
    .drop("row_no")
)

4. `rank()` to identify top products inside each category.

In [0]:
product_sales_df = (
    silver_products_df.join(silver_sales_orders_df, "product_id", "inner")
    .groupBy("category", "product_id", "product_name")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales"),
    )
)

rank_window = Window.partitionBy("category").orderBy(F.desc("total_sales"))

top_products_df = (
    product_sales_df
    .withColumn("rank", F.rank().over(rank_window))
    .filter(F.col("rank") <= 5)
    .orderBy("category", F.desc("total_sales"))
    .drop("rank")
)

display(top_products_df)

5. `dense_rank()` to rank customers within each state.


In [0]:
# Ranking by loyalty_points
rank_window = Window.partitionBy("state").orderBy(F.desc("loyalty_points"))
display(silver_customers_df.withColumn("rank", F.rank().over(rank_window)))

6. A running revenue total by category and month.

In [0]:
display(
    silver_sales_orders_df.join(silver_products_df, "product_id", "inner")
    .withColumn(
        "running_total_revenue",
        F.sum("net_sales").over(
            Window.partitionBy("category")
            .orderBy("order_year", "order_month")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        ),
    )
    .withColumn("running_total_revenue", F.round(F.col("running_total_revenue"), 2))
    .select(
        F.col("order_year"),
        F.col("order_month"),
        F.col("category"),
        F.col("net_sales"),
        F.col("running_total_revenue"),
    )
)

7. The latest order for every customer.

In [0]:
display(
    silver_sales_orders_df.join(silver_customers_df, "customer_id", "inner")
    .withColumn(
        "rank",
        F.rank().over(
            Window.partitionBy("customer_id").orderBy(F.desc("order_timestamp"))
        ),
    )
    .filter(F.col("rank") == 1)
    .drop("rank")
)